# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [7]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "indonesia_palm_oil_v1_2_5.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_bersih.parquet"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH}. "
        "Pastikan dataset berada di data/raw/."
    )

df = pl.read_csv(DATA_PATH)

print(f"Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom")
df.head()

Ukuran awal: 1,694,907 baris x 34 kolom


year,country_of_production,product_type,forest_500_palm_oil,zero_deforestation_indonesia_palm_oil,province_of_production,kabupaten_of_production,mill,mill_trase_id,mill_group,refinery,refinery_trase_id,refinery_group,exporter,exporter_group,port_of_export,importer,importer_group,country_of_first_import,economic_bloc,province_of_production_trase_id,kabupaten_of_production_trase_id,country_of_production_trase_id,country_of_first_import_trase_id,palm_oil_deforestation_10_year_total_exposure,palm_oil_deforestation_annual_exposure,total_emission_exposure,net_emission_luc_exposure,emission_subsidence_exposure,gross_emission_luc_exposure,emissions_fire_on_peat_exposure,volume,oil_palm_ha,fob
i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,str
2013,"""INDONESIA""","""PALM OIL""","""NOT ASSESSED""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-MILL-X""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-REFINERY-X""","""UNKNOWN""","""ACCESS BALI""","""ACCESS BALI""","""TANJUNG PRIOK""","""JOV-HENTON LTD""","""JOV-HENTON LTD""","""NIGERIA""","""NIGERIA""","""ID-XX""","""ID-XXXX""","""ID""","""NG""",null,null,null,null,null,null,null,22.8,null,null
2013,"""INDONESIA""","""PALM OIL""","""NOT ASSESSED""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-MILL-X""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-REFINERY-X""","""UNKNOWN""","""ADEI PLANTATION & INDUSTRY""","""KUALA LUMPUR KEPONG (KLK)""","""DUMAI""","""ACALPO""","""ACALPO""","""INDIA""","""INDIA""","""ID-XX""","""ID-XXXX""","""ID""","""IN""",null,null,null,null,null,null,null,3000.0,null,null
2013,"""INDONESIA""","""PALM OIL""","""NOT ASSESSED""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-MILL-X""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-REFINERY-X""","""UNKNOWN""","""ADEI PLANTATION & INDUSTRY""","""KUALA LUMPUR KEPONG (KLK)""","""DUMAI""","""BUNGE""","""BUNGE""","""INDIA""","""INDIA""","""ID-XX""","""ID-XXXX""","""ID""","""IN""",null,null,null,null,null,null,null,1000.0,null,null
2013,"""INDONESIA""","""PALM OIL""","""NOT ASSESSED""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-MILL-X""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-REFINERY-X""","""UNKNOWN""","""ADEI PLANTATION & INDUSTRY""","""KUALA LUMPUR KEPONG (KLK)""","""DUMAI""","""CARGILL""","""CARGILL""","""INDIA""","""INDIA""","""ID-XX""","""ID-XXXX""","""ID""","""IN""",null,null,null,null,null,null,null,1000.0,null,null
2013,"""INDONESIA""","""PALM OIL""","""NOT ASSESSED""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-MILL-X""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-REFINERY-X""","""UNKNOWN""","""ADEI PLANTATION & INDUSTRY""","""KUALA LUMPUR KEPONG (KLK)""","""DUMAI""","""CARGILL""","""CARGILL""","""TANZANIA""","""TANZANIA""","""ID-XX""","""ID-XXXX""","""ID""","""TZ""",null,null,null,null,null,null,null,500.0,null,null


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [8]:
connection = duckdb.connect()
connection.register('dataset', df)

profil = connection.execute('SUMMARIZE SELECT * FROM dataset').pl()
profil

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""year""","""BIGINT""","""2013""","""2022""",11,"""2020.0519167128343""","""1.3540499690542493""","""2019""","""2020""","""2021""",1694907,0.00
"""country_of_production""","""VARCHAR""","""INDONESIA""","""INDONESIA""",1,null,null,null,null,null,1694907,0.00
"""product_type""","""VARCHAR""","""PALM OIL""","""REFINED PALM OIL""",2,null,null,null,null,null,1694907,0.00
"""forest_500_palm_oil""","""VARCHAR""","""0""","""NOT ASSESSED""",7,null,null,null,null,null,1694907,0.00
"""zero_deforestation_indonesia_p…","""VARCHAR""","""COMPANY COMMITMENT""","""UNKNOWN""",4,null,null,null,null,null,1694907,0.00
…,…,…,…,…,…,…,…,…,…,…,…
"""gross_emission_luc_exposure""","""VARCHAR""","""0""","""999.9926319638474""",142065,null,null,null,null,null,1694907,60.49
"""emissions_fire_on_peat_exposur…","""VARCHAR""","""0""","""997.5440867831901""",51306,null,null,null,null,null,1694907,60.49
"""volume""","""DOUBLE""","""5.041356112552372e-11""","""11605224.752096057""",1619108,"""226.34659706788156""","""15280.810851364458""","""0.15306361733860988""","""2.1315341458082173""","""21.312387315890017""",1694907,0.00


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [9]:
jumlah_baris = len(df)

null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null')
    )
    .filter(pl.col('jumlah_null') > 0)
    .sort('jumlah_null', descending=True)
)

null_summary

kolom,jumlah_null,persen_null
str,u32,f64
"""total_emission_exposure""",1025290,60.49
"""net_emission_luc_exposure""",1025290,60.49
"""emission_subsidence_exposure""",1025290,60.49
"""gross_emission_luc_exposure""",1025290,60.49
"""emissions_fire_on_peat_exposur…",1025290,60.49
"""fob""",10957,0.65
"""palm_oil_deforestation_10_year…",10233,0.6
"""palm_oil_deforestation_annual_…",10233,0.6
"""oil_palm_ha""",10233,0.6


In [11]:
for col in null_summary["kolom"].to_list():
    print(f"\n=== {col} ===")
    
    result = (
        df
        .select(
            pl.col(col)
            .drop_nulls()
            .value_counts(sort=True)
        )
        .unnest(col)
        .head(5)
    )
    
    print(result)


=== total_emission_exposure ===
shape: (5, 2)
┌─────────────────────────┬────────┐
│ total_emission_exposure ┆ count  │
│ ---                     ┆ ---    │
│ str                     ┆ u32    │
╞═════════════════════════╪════════╡
│ 0                       ┆ 357087 │
│ 46.8391801257788        ┆ 20     │
│ 3.23367089441404        ┆ 20     │
│ 0.0906605446796969      ┆ 19     │
│ 0.0590438624974629      ┆ 19     │
└─────────────────────────┴────────┘

=== net_emission_luc_exposure ===
shape: (5, 2)
┌───────────────────────────┬────────┐
│ net_emission_luc_exposure ┆ count  │
│ ---                       ┆ ---    │
│ str                       ┆ u32    │
╞═══════════════════════════╪════════╡
│ 0                         ┆ 508824 │
│ 0.0590438624974629        ┆ 19     │
│ 0.00202770800955938       ┆ 19     │
│ 0.0706685578296504        ┆ 19     │
│ -0.01506325122609278      ┆ 19     │
└───────────────────────────┴────────┘

=== emission_subsidence_exposure ===
shape: (5, 2)
┌───────────────

## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [12]:
# numeric columns yang tersimpan sebagai string pada dataset
kolom_numerik_string = [
    "palm_oil_deforestation_10_year_total_exposure",
    "palm_oil_deforestation_annual_exposure",
    "total_emission_exposure",
    "net_emission_luc_exposure",
    "emission_subsidence_exposure",
    "gross_emission_luc_exposure",
    "emissions_fire_on_peat_exposure",
    "oil_palm_ha",
    "fob",
]

df_missing_handled = df.with_columns(
    [
        pl.col(col).cast(pl.Float64, strict=False)
        for col in kolom_numerik_string
    ]
)


kolom_imputasi_median = [
    "oil_palm_ha",
    "fob",
]

df_missing_handled = df_missing_handled.with_columns(
    [
        pl.col(col).fill_null(pl.col(col).median())
        for col in kolom_imputasi_median
    ]
)

print(
    f"Null sebelum: "
    f"{df.null_count().sum_horizontal().sum():,}"
)

print(
    f"Null sesudah: "
    f"{df_missing_handled.null_count().sum_horizontal().sum():,}"
)

Null sebelum: 5,168,106
Null sesudah: 5,146,916


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [13]:
# TODO: ganti 'kolom_numerik' dengan kolom numerik utama pada dataset Anda
KOLOM_NUMERIK = "volume"

# Hitung kuartil
q1 = df_missing_handled[KOLOM_NUMERIK].quantile(0.25)
q3 = df_missing_handled[KOLOM_NUMERIK].quantile(0.75)

# Hitung IQR
iqr = q3 - q1

# Batas outlier berdasarkan metode IQR
batas_bawah_iqr = q1 - 1.5 * iqr
batas_atas = q3 + 1.5 * iqr

batas_bawah = max(0, batas_bawah_iqr)

print(f"Kolom numerik: {KOLOM_NUMERIK}")
print(f"Q1: {q1}")
print(f"Q3: {q3}")
print(f"IQR: {iqr}")
print(f"Batas bawah IQR: {batas_bawah_iqr}")
print(f"Batas bawah final: {batas_bawah}")
print(f"Batas atas: {batas_atas}")

jumlah_outlier = df_missing_handled.filter(
    (pl.col(KOLOM_NUMERIK) < batas_bawah)
    | (pl.col(KOLOM_NUMERIK) > batas_atas)
).height

print(f"Jumlah outlier: {jumlah_outlier:,}")

Kolom numerik: volume
Q1: 0.1521182186311248
Q3: 21.21685539046564
IQR: 21.064737171834516
Batas bawah IQR: -31.44498753912065
Batas bawah final: 0
Batas atas: 52.813961148217416
Jumlah outlier: 281,597


In [14]:
# Penanganan outlier dengan winsorization

df_outlier_handled = df_missing_handled.with_columns(
    pl.col(KOLOM_NUMERIK).clip(
        lower_bound=batas_bawah,
        upper_bound=batas_atas
    )
)

print("Penanganan outlier selesai.")
print(f"Jumlah baris: {len(df_outlier_handled):,}")

Penanganan outlier selesai.
Jumlah baris: 1,694,907


In [15]:
# Bandingkan nilai volume sebelum dan sesudah winsorization

jumlah_nilai_berubah = (
    df_missing_handled
    .select(
        (
            pl.col(KOLOM_NUMERIK)
            != df_outlier_handled[KOLOM_NUMERIK]
        )
        .sum()
        .alias("jumlah_nilai_berubah")
    )
    .item()
)

print(f"Nilai {KOLOM_NUMERIK} yang berubah: {jumlah_nilai_berubah:,}")

print("\nStatistik sebelum penanganan:")
print(
    df_missing_handled
    .select(
        pl.col(KOLOM_NUMERIK).min().alias("min"),
        pl.col(KOLOM_NUMERIK).max().alias("max")
    )
)

print("\nStatistik sesudah penanganan:")
print(
    df_outlier_handled
    .select(
        pl.col(KOLOM_NUMERIK).min().alias("min"),
        pl.col(KOLOM_NUMERIK).max().alias("max")
    )
)

Nilai volume yang berubah: 281,597

Statistik sebelum penanganan:
shape: (1, 2)
┌────────────┬──────────┐
│ min        ┆ max      │
│ ---        ┆ ---      │
│ f64        ┆ f64      │
╞════════════╪══════════╡
│ 5.0414e-11 ┆ 1.1605e7 │
└────────────┴──────────┘

Statistik sesudah penanganan:
shape: (1, 2)
┌────────────┬───────────┐
│ min        ┆ max       │
│ ---        ┆ ---       │
│ f64        ┆ f64       │
╞════════════╪═══════════╡
│ 5.0414e-11 ┆ 52.813961 │
└────────────┴───────────┘


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [17]:
kandidat_kunci = [
    "mill_trase_id",
    "refinery_trase_id",
    "exporter",
    "importer",
    "country_of_first_import",
]

for kolom in kandidat_kunci:
    jumlah_unik = df_outlier_handled[kolom].n_unique()
    jumlah_baris = len(df_outlier_handled)
    
    print(
        f"{kolom}: "
        f"{jumlah_unik:,} nilai unik dari {jumlah_baris:,} baris"
    )

mill_trase_id: 1,209 nilai unik dari 1,694,907 baris
refinery_trase_id: 67 nilai unik dari 1,694,907 baris
exporter: 492 nilai unik dari 1,694,907 baris
importer: 2,737 nilai unik dari 1,694,907 baris
country_of_first_import: 183 nilai unik dari 1,694,907 baris


In [18]:
# Composite business key untuk mengidentifikasi satu record supply chain
KOLOM_KUNCI = [
    "year",
    "product_type",
    "mill_trase_id",
    "refinery_trase_id",
    "exporter",
    "importer",
    "country_of_first_import",
    "port_of_export",
]

total_baris = len(df_outlier_handled)

baris_unik = (
    df_outlier_handled
    .select(KOLOM_KUNCI)
    .unique()
    .height
)

jumlah_duplikat = total_baris - baris_unik

print(f"Total baris: {total_baris:,}")
print(f"Baris unik berdasarkan composite key: {baris_unik:,}")
print(f"Duplikat berdasarkan composite key: {jumlah_duplikat:,}")

Total baris: 1,694,907
Baris unik berdasarkan composite key: 617,082
Duplikat berdasarkan composite key: 1,077,825


In [19]:
jumlah_baris_asli = len(df_outlier_handled)

jumlah_baris_unik_lengkap = (
    df_outlier_handled
    .unique()
    .height
)

jumlah_duplikat_lengkap = (
    jumlah_baris_asli - jumlah_baris_unik_lengkap
)

print(f"Total baris: {jumlah_baris_asli:,}")
print(f"Baris unik seluruh kolom: {jumlah_baris_unik_lengkap:,}")
print(f"Duplikat identik seluruh kolom: {jumlah_duplikat_lengkap:,}")

Total baris: 1,694,907
Baris unik seluruh kolom: 1,694,907
Duplikat identik seluruh kolom: 0


In [21]:
# membuat identifier unik untuk setiap record karena dataset tidak memiliki kolom ID unik bawaan.
df_outlier_handled = df_outlier_handled.with_row_index(
    name="id_unik",
    offset=1
)

print(f"Jumlah baris: {len(df_outlier_handled):,}")
print(f"Jumlah ID unik: {df_outlier_handled['id_unik'].n_unique():,}")

Jumlah baris: 1,694,907
Jumlah ID unik: 1,694,907


In [22]:
# TODO: ganti 'id_unik' dengan kolom kunci unik pada dataset Anda
KOLOM_KUNCI = 'id_unik'

total_baris = len(df_outlier_handled)
baris_unik = df_outlier_handled[KOLOM_KUNCI].n_unique() if KOLOM_KUNCI in df_outlier_handled.columns else total_baris
print(f'Total baris: {total_baris:,}')
print(f'Baris unik berdasarkan {KOLOM_KUNCI}: {baris_unik:,}')
print(f'Duplikat: {total_baris - baris_unik:,}')

df_dedup = df_outlier_handled.unique(
    subset=[KOLOM_KUNCI] if KOLOM_KUNCI in df_outlier_handled.columns else None,
    keep='first',
    maintain_order=True,
)

Total baris: 1,694,907
Baris unik berdasarkan id_unik: 1,694,907
Duplikat: 0


## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [23]:
connection.register('dataset_bersih', df_dedup)

# TODO: sesuaikan aturan validasi dengan kolom dan batas nilai pada dataset Anda
validasi = connection.execute(f"""
    SELECT
        COUNT(*) AS total_baris,
        SUM(CASE WHEN {KOLOM_NUMERIK} IS NULL THEN 1 ELSE 0 END) AS pelanggaran_null,
        SUM(CASE WHEN {KOLOM_NUMERIK} < {batas_bawah} OR {KOLOM_NUMERIK} > {batas_atas}
            THEN 1 ELSE 0 END) AS pelanggaran_batas
    FROM dataset_bersih
""").pl()

validasi

total_baris,pelanggaran_null,pelanggaran_batas
i64,"decimal[38,0]","decimal[38,0]"
1694907,0,0


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [24]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH)

print(f'Baris sebelum cleaning: {len(df):,}')
print(f'Baris sesudah cleaning: {len(df_dedup):,}')
print(f'Data bersih tersimpan di: {OUTPUT_PATH}')

Baris sebelum cleaning: 1,694,907
Baris sesudah cleaning: 1,694,907
Data bersih tersimpan di: /home/jovyan/work/data/processed/dataset_bersih.parquet


## Catatan Milestone 2

Dokumentasikan di sini: dimensi kualitas data yang paling bermasalah, strategi yang dipilih untuk missing values dan outlier, jumlah baris yang berubah, serta aturan validasi yang diterapkan.

### 1. Dimensi kualitas data yang paling bermasalah
Masalah kualitas data yang paling menonjol adalah missing values pada kolom exposure emisi. 
Lima kolom, yaitu `total_emission_exposure`, `net_emission_luc_exposure`,
`emission_subsidence_exposure`, `gross_emission_luc_exposure`, dan
`emissions_fire_on_peat_exposure`, masing-masing memiliki 1.025.290 nilai null
atau sekitar 60,49% dari total 1.694.907 baris.

Selain itu, terdapat missing values pada `fob` sebanyak 10.957 baris,
`oil_palm_ha` sebanyak 10.233 baris, serta dua kolom deforestation exposure
masing-masing sebanyak 10.233 baris.

### 2. Strategi missing values
Kolom `oil_palm_ha` dan `fob` dikonversi ke tipe numerik dan nilai null
diisi menggunakan median masing-masing kolom.

Kolom exposure emisi yang memiliki missing values dalam jumlah besar
tidak diimputasi karena proporsinya mencapai sekitar 60,49% dan tidak ada
dasar yang cukup untuk menggantikan nilai tersebut secara arbitrer.

### 3. Strategi outlier
Outlier diperiksa pada kolom `volume` menggunakan metode Interquartile Range (IQR).
Batas bawah IQR yang diperoleh bernilai negatif sehingga batas bawah akhir
ditetapkan menjadi 0, sedangkan batas atas adalah 52,813961.

Terdapat 281.597 nilai `volume` yang berada di luar batas IQR.
Outlier ditangani menggunakan clipping ke rentang 0 sampai 52,813961,
bukan dengan menghapus baris.

### 4. Perubahan jumlah baris
Jumlah baris sebelum cleaning adalah 1.694.907 dan jumlah baris setelah
cleaning tetap 1.694.907.

Tidak ada baris yang dihapus pada proses cleaning. Sebanyak 281.597 nilai
`volume` mengalami perubahan akibat proses clipping.

Pada pemeriksaan duplikasi, tidak ditemukan exact duplicate pada seluruh
kolom dataset.

### 5. Aturan validasi
Validasi dasar diterapkan pada kolom `volume`, yaitu:

1. `volume` tidak boleh bernilai NULL.
2. `volume` harus berada pada rentang batas yang telah ditetapkan,
   yaitu 0 sampai 52,813961.

Hasil validasi digunakan untuk memastikan data bersih memenuhi batas nilai
yang telah ditentukan setelah proses cleaning.